In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1999
month = 9


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1999-09-30


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1999-09-01 12:00:00
end_date 1999-09-02 12:00:00
start_date 1999-09-03 12:00:00
end_date 1999-09-04 12:00:00
start_date 1999-09-05 12:00:00
end_date 1999-09-06 12:00:00
start_date 1999-09-07 12:00:00
end_date 1999-09-08 12:00:00
start_date 1999-09-09 12:00:00
end_date 1999-09-10 12:00:00
start_date 1999-09-11 12:00:00
end_date 1999-09-12 12:00:00
start_date 1999-09-13 12:00:00
end_date 1999-09-14 12:00:00
start_date 1999-09-15 12:00:00
end_date 1999-09-16 12:00:00
start_date 1999-09-17 12:00:00
end_date 1999-09-18 12:00:00
start_date 1999-09-19 12:00:00
end_date 1999-09-20 12:00:00
start_date 1999-09-21 12:00:00
end_date 1999-09-22 12:00:00
start_date 1999-09-23 12:00:00
end_date 1999-09-24 12:00:00
start_date 1999-09-25 12:00:00
end_date 1999-09-26 12:00:00
start_date 1999-09-27 12:00:00
end_date 1999-09-28 12:00:00
start_date 1999-09-29 12:00:00
end_date 1999-09-30 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [01:49<25:34, 109.60s/it]

 13%|██████▋                                           | 2/15 [02:09<12:21, 57.05s/it]

 20%|██████████                                        | 3/15 [02:32<08:14, 41.20s/it]

 27%|█████████████▎                                    | 4/15 [02:53<06:06, 33.34s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:01<00:41, 20.92s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [06:20<00:20, 20.20s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:41<00:00, 20.54s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:41<00:00, 26.77s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_1999-09.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [02:01<28:25, 121.82s/it]

 13%|██████▋                                           | 2/15 [02:22<13:31, 62.42s/it]

 20%|██████████                                        | 3/15 [02:42<08:35, 42.92s/it]

 27%|█████████████▎                                    | 4/15 [02:59<06:00, 32.75s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [06:13<00:43, 21.90s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [06:30<00:20, 20.60s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:50<00:00, 20.32s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:50<00:00, 27.37s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_1999-09.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                             | 1/15 [02:06<29:35, 126.81s/it]

 13%|██████▋                                           | 2/15 [02:26<13:52, 64.07s/it]

 20%|██████████                                        | 3/15 [02:50<09:08, 45.73s/it]

 27%|█████████████▎                                    | 4/15 [03:17<07:01, 38.35s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [07:38<01:14, 37.44s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [07:56<00:31, 31.51s/it]

100%|█████████████████████████████████████████████████| 15/15 [08:16<00:00, 28.25s/it]

100%|█████████████████████████████████████████████████| 15/15 [08:16<00:00, 33.12s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_1999-09.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                              | 1/15 [00:55<12:55, 55.36s/it]

 13%|██████▋                                           | 2/15 [02:01<13:23, 61.79s/it]

 20%|██████████                                        | 3/15 [02:21<08:34, 42.85s/it]

 27%|█████████████▎                                    | 4/15 [02:40<06:05, 33.20s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [05:43<00:43, 21.52s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [06:11<00:23, 23.46s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:35<00:00, 23.62s/it]

100%|█████████████████████████████████████████████████| 15/15 [06:35<00:00, 26.38s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_1999-09.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                          | 0/15 [00:00<?, ?it/s]

  7%|███▎                                              | 1/15 [00:17<04:09, 17.85s/it]

 13%|██████▋                                           | 2/15 [00:36<03:57, 18.30s/it]

 20%|██████████                                        | 3/15 [01:18<05:47, 28.97s/it]

 27%|█████████████▎                                    | 4/15 [01:37<04:36, 25.16s/it]

...

 87%|██████████████████████████████████████████▍      | 13/15 [04:38<00:39, 19.72s/it]

 93%|█████████████████████████████████████████████▋   | 14/15 [04:56<00:19, 19.30s/it]

100%|█████████████████████████████████████████████████| 15/15 [05:14<00:00, 18.81s/it]

100%|█████████████████████████████████████████████████| 15/15 [05:14<00:00, 20.98s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_1999-09.nc
